Cell 0 — Parameters (Papermill tag: parameters)
This cell is tagged parameters in Jupyter so Papermill can inject overrides here at runtime. That tag is set in the notebook metadata, not in the code itself — in JupyterLab: right-click the cell → Add Tag → type parameters.

In [1]:
# Cell 0 — tagged: parameters
# Papermill injects overrides here. Defaults match scad/params.json.
# To run a sweep: papermill 01_geometry_openscad.ipynb out.ipynb -p part_name "sweep_01"
import os
os.chdir("/workspace")

import sys
sys.path.insert(0, "/workspace")  # ensure src/ is importable inside Docker

PARAMS_JSON   = "scad/params.json"
SCAD_FILE     = "scad/base_part.scad"
TIMEOUT_S     = 120
PART_NAME_OVERRIDE = None  # set by Papermill e.g. "bracket_narrow"

Cell 1 — Load and validate params

In [2]:
# Cell 1 — Load params and validate schema
from src.geometry.param_schema import PipelineParams

params = PipelineParams.from_json(PARAMS_JSON)

# Apply Papermill override if provided
if PART_NAME_OVERRIDE is not None:
    params.part_name = PART_NAME_OVERRIDE

# Validate — raises AssertionError with a clear message if anything is wrong
params.validate()

print(f"part_name:    {params.part_name}")
print(f"geometry:     {params.geometry}")
print(f"mesh_hints:   {params.mesh_hints}")
print(f"load_hints:   {params.load_hints}")
print(f"export dir:   {params.export.stl_output_dir}")

part_name:    base_part
geometry:     GeometryParams(length=100.0, width=60.0, height=20.0, wall_thickness=4.0, fillet_radius=2.0, mounting_hole_diameter=6.0, mounting_hole_inset=10.0)
mesh_hints:   MeshHints(target_element_size=8.0, opt_domain_element_size_mm=2.5, refinement_regions=[])
load_hints:   LoadHints(primary_face='top', load_magnitude_n=10000.0)
export dir:   outputs/meshes


Cell 2 — Show OpenSCAD defines
Inspectable checkpoint — you can verify exactly what flags will be passed to OpenSCAD before running it.

In [3]:
# Cell 2 — Inspect defines before running OpenSCAD
defines = params.to_openscad_defines()
print("OpenSCAD -D defines:")
for k, v in defines.items():
    print(f"  {k:<30} = {v}")

OpenSCAD -D defines:
  LENGTH                         = 100.0
  WIDTH                          = 60.0
  HEIGHT                         = 20.0
  WALL_THICKNESS                 = 4.0
  FILLET_RADIUS                  = 2.0
  MOUNTING_HOLE_DIAMETER         = 6.0
  MOUNTING_HOLE_INSET            = 10.0


Cell 3 — Run OpenSCAD

In [4]:
# Cell 3 — Compile geometry
from src.geometry.openscad_runner import run_openscad

result = run_openscad(params, scad_file=SCAD_FILE, timeout_s=TIMEOUT_S)

print(f"Success:    {result.success}")
print(f"Duration:   {result.duration_s}s")
print(f"STL path:   {result.stl_path}")
print(f"STL size:   {result.stl_size_kb} KB")
if result.stderr:
    print(f"\nStderr:\n{result.stderr}")

# Hard fail here — don't let a bad STL silently propagate to Stage 2
result.raise_if_failed()

Success:    True
Duration:   0.27s
STL path:   outputs/meshes/base_part.stl
STL size:   101.82 KB

Stderr:
Geometries in cache: 10
Geometry cache size in bytes: 40976
CGAL Polyhedrons in cache: 2
CGAL cache size in bytes: 799088
Total rendering time: 0:00:00.147
   Top level object is a 3D object:
   Simple:        yes
   Vertices:      336
   Halfedges:    1008
   Edges:         504
   Halffacets:    342
   Facets:        171
   Volumes:         2



Cell 4 — Quick mesh preview
Renders headlessly via PyVista and saves a PNG to outputs/reports/. This gives you a visual sanity check without needing a display.

In [5]:
# Cell 4 — Headless STL preview → outputs/reports/
import os, warnings
import numpy as np
import meshio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path

warnings.filterwarnings(
    "ignore",
    message="overflow encountered in scalar multiply",
    category=RuntimeWarning,
    module=r"meshio\.stl\._stl",
)

m     = meshio.read(str(result.stl_path))
verts = m.points[m.cells_dict["triangle"]]  # (N, 3, 3)
pts   = verts.reshape(-1, 3)

print(f"Vertices:  {len(m.points)}")
print(f"Faces:     {len(m.cells_dict['triangle'])}")
print(f"Bounds:    {[round(v, 2) for v in [pts[:,0].min(), pts[:,0].max(), pts[:,1].min(), pts[:,1].max(), pts[:,2].min(), pts[:,2].max()]]}")

report_dir = Path("outputs/reports")
report_dir.mkdir(parents=True, exist_ok=True)
png_path = report_dir / f"{params.part_name}_geometry.png"

fig = plt.figure(figsize=(7, 5))
ax  = fig.add_subplot(111, projection="3d")
ax.add_collection3d(
    Poly3DCollection(verts, alpha=0.6, facecolor="lightsteelblue", edgecolor="grey", linewidth=0.1)
)
for dim, setter in zip(range(3), [ax.set_xlim, ax.set_ylim, ax.set_zlim]):
    setter(pts[:, dim].min(), pts[:, dim].max())
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.set_title(params.part_name)
plt.tight_layout()
plt.savefig(str(png_path), dpi=150)
plt.close()

print(f"\nPreview saved: {png_path}")

Vertices:  336
Faces:     684
Bounds:    [np.float64(-2.0), np.float64(98.0), np.float64(-2.0), np.float64(58.0), np.float64(0.0), np.float64(20.0)]

Preview saved: outputs/reports/base_part_geometry.png


Cell 5 — Export stage output for handoff
Writes a small JSON sidecar next to the STL so Stage 2 (02_mesh_gmsh.ipynb) knows where to find the geometry and what parameters were used, without having to re-read params.json.

In [6]:
# Cell 5 — Write stage handoff sidecar
# 02_mesh_gmsh.ipynb reads this to find the STL and mesh hints
import json
from pathlib import Path
from dataclasses import asdict

handoff = {
    "stage":        "01_geometry",
    "part_name":    params.part_name,
    "stl_path":     str(result.stl_path),
    "stl_size_kb":  result.stl_size_kb,
    "duration_s":   result.duration_s,
    "mesh_hints":   asdict(params.mesh_hints),
    "load_hints":   asdict(params.load_hints),
}

handoff_path = Path(params.export.stl_output_dir) / f"{params.part_name}_stage01.json"
handoff_path.write_text(json.dumps(handoff, indent=2))
print(f"Handoff written: {handoff_path}")
print(json.dumps(handoff, indent=2))

Handoff written: outputs/meshes/base_part_stage01.json
{
  "stage": "01_geometry",
  "part_name": "base_part",
  "stl_path": "outputs/meshes/base_part.stl",
  "stl_size_kb": 101.82,
  "duration_s": 0.27,
  "mesh_hints": {
    "target_element_size": 8.0,
    "opt_domain_element_size_mm": 2.5,
    "refinement_regions": []
  },
  "load_hints": {
    "primary_face": "top",
    "load_magnitude_n": 10000.0
  }
}


How these files connect to the rest of the pipeline
The sidecar JSON from Cell 5 is the contract between Stage 1 and Stage 2. When 02_mesh_gmsh.ipynb runs, its Cell 0 reads outputs/meshes/<part_name>_stage01.json rather than re-parsing params.json — this means even if params change mid-run, Stage 2 always meshes the geometry that was actually compiled, not whatever the current params say.
The mesh_hints.target_element_size value travels through this sidecar into src/meshing/gmsh_pipeline.py in Stage 2, where it controls the global mesh size. The load_hints travel all the way to src/fea/boundary_conditions.py in Stage 3.
Ready to move to Stage 2 — 02_mesh_gmsh.ipynb and src/meshing/?